In [ ]:
LATITUDE = None
LONGITUDE = None
RADIUS_MILES = None

In [ ]:
dataset = "data\\UsageData-11_27_2024-03_31_2026-clean.csv"

import pandas as pd
hourly_usage = pd.read_csv(dataset) 
hourly_usage["timestamp"] = pd.to_datetime(hourly_usage["timestamp"], utc=True).dt.tz_convert("America/New_York").dt.tz_localize(None)
hourly_usage

In [ ]:
import meteostat

point = meteostat.Point(LATITUDE, LONGITUDE)
nearby = meteostat.stations.nearby(point, radius=1609.344 * RADIUS_MILES)

start = pd.Timestamp(hourly_usage["timestamp"].min()).to_pydatetime()
end = pd.Timestamp(hourly_usage["timestamp"].max()).to_pydatetime()

# Find the closest station with sufficient hourly coverage for temperature
station = None
for station_id in nearby.index:
    completeness = meteostat.hourly(station_id, start, end, parameters=[meteostat.parameters.Parameter.TEMP]).completeness()
    if completeness is not None and completeness >= 0.9:
        station = station_id
        break

if station is None:
    raise RuntimeError("No nearby station found with sufficient hourly coverage")

print(f"Selected Station: {station} ({nearby.loc[station, 'name']}, completeness={completeness:.0%})")

In [ ]:
observations = meteostat.hourly(station, start, end).fetch()

# Compute wind chill
# T_wc = 13.12 + 0.6215 * T_a * (0.3965 * T_a - 11.37) * v^0.16
observations["wchill"] = 13.12 + (0.6215*observations["temp"]) - (11.37*observations["wspd"]**0.16) + (0.3965*observations["temp"]*observations["wspd"]**.16)

observations.to_csv(f"data\\meteorological_observations_{start.month}_{start.day}_{start.year}-{end.month}_{end.day}_{end.year}.csv")